# Chapter 2: Routing and URL Building 🗺️

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='1. getting_started.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Chapter 1: Getting Started</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='3. templates_with_jinja2.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 3: Jinja2 Templates →</a>
</div>

---

## 🎯 The Big Picture

When you type a URL in your browser, how does Flask know which Python function to call? **Routing** is the map that connects URLs to code. Get this right and your app's structure will be clean and logical forever.

Think of your Flask app as a city and URLs as street addresses — routing is the GPS that gets every request to the right destination.

> 💡 **Tip:** Good URL design is a gift to your future self (and your users). Clear, predictable URLs make apps easier to use, share, and maintain.

---

## 🧠 Core Concepts: The "Why"

### What Is a Route?

A **route** is a mapping between a URL pattern and a Python function (called a **view function**). When Flask receives an HTTP request, it looks up the URL in its routing table and calls the matching function.

> 📞 **Analogy:** Routes are like a phone directory — a URL is the name you look up, the function is the phone number you get. You give Flask a URL, it gives you back the right code to run.

### The `@app.route()` Decorator

A **decorator** is a special Python syntax (the `@` symbol) that *wraps* a function with extra behaviour — without changing the function itself. `@app.route('/hello')` tells Flask: *"Whenever someone visits `/hello`, call the function below."*

```python
@app.route('/hello')        # <- decorator registers this URL
def hello():                 # <- the view function Flask will call
    return 'Hello, World!'
```

Flask stores all these mappings in an internal **URL map** — you can inspect it with `app.url_map`.

**Important:** `@app.route()` is *syntactic sugar* for `app.add_url_rule()`. These two snippets are exactly equivalent:

```python
# Decorator style (idiomatic Flask)
@app.route('/hello')
def hello():
    return 'Hello!'

# Explicit style (same result — useful when building routes programmatically)
def hello():
    return 'Hello!'
app.add_url_rule('/hello', endpoint='hello', view_func=hello)
```

The `endpoint` is the internal name Flask uses to look up a route when you call `url_for('hello')`. By default it equals the function name, but you can customise it with `@app.route('/hello', endpoint='my_custom_name')`.

### URL Variables & Type Converters

URLs don't have to be static! You can embed **variable segments** using angle brackets: `<variable_name>`. Flask extracts the value and passes it as a keyword argument to your view function.

Add a **type converter** before the variable name to restrict what values are accepted and auto-convert the type:

| Converter | Syntax | Accepts | Python type | Example URL |
|-----------|--------|---------|-------------|-------------|
| string | `<string:name>` | Any text without `/` (default) | `str` | `/user/alice` |
| int | `<int:id>` | Positive integers only | `int` | `/post/42` |
| float | `<float:score>` | Positive floating-point numbers | `float` | `/rate/9.75` |
| path | `<path:subpath>` | Text **including** `/` slashes | `str` | `/files/docs/api/v2` |
| uuid | `<uuid:token>` | UUID-format strings (e.g. `550e8400-e29b-41d4-a716-446655440000`) | `uuid.UUID` | `/token/550e8400-...` |

> 💡 **Tip:** Always use the right converter! `<int:id>` means `/users/abc` returns 404 automatically — Flask won't even call your function with a non-integer.

**Why type converters matter:**
- **Security:** They prevent type-confusion bugs. Without `<int:id>`, a route like `SELECT * FROM posts WHERE id = '{id}'` could be fed a string containing SQL.
- **Clean URLs:** `/posts/42` is cleaner than `/posts?id=42` for identifying resources.
- **Free validation:** A 404 for a mismatched type is much better than a 500 from a casting error inside your view function.

**The `path` converter** is special: it matches slashes, so `/files/docs/api/v2` passes `'docs/api/v2'` to your function. Use it for file-path style URLs.

**The `uuid` converter** is useful for non-guessable resource identifiers. It rejects any value that is not a valid UUID, ensuring `uuid.UUID` objects reach your view function type-safe.

### HTTP Methods

Every HTTP request has a **method** (sometimes called a *verb*) that signals the intended action. By default, Flask routes only accept `GET` requests. You can allow others via the `methods` parameter.

| Method | Purpose | Has request body? | Typical use | Idempotent? |
|--------|---------|-------------------|-------------|-------------|
| `GET` | Retrieve a resource | No | Load a page, fetch data | Yes |
| `POST` | Submit data / create resource | Yes | Submit a form, create a record | No |
| `PUT` | Replace a resource entirely | Yes | Full update of a record | Yes |
| `PATCH` | Partially update a resource | Yes | Change one field of a record | Yes |
| `DELETE` | Remove a resource | Optional | Delete a record | Yes |

**Idempotent** means calling the operation multiple times gives the same result as calling it once. `GET /posts/42` ten times returns the same post ten times — safe to retry. `POST /posts` ten times creates ten posts — not idempotent.

**Key rules:**
- `GET` requests should **never** change server state. If clicking a link could delete something, that's a bug waiting to happen.
- `POST` is for *creating* or *submitting*. It's the correct method for HTML form submissions.
- `PUT` *replaces* the entire resource. `PATCH` changes only the fields you specify.
- `DELETE` removes the resource. A second `DELETE` on the same resource returns 404 — but that's still considered idempotent because the end state (resource gone) is the same.

> 💡 **Common beginner trap:** HTML forms only support `GET` and `POST` natively. To use `PUT`, `PATCH`, or `DELETE` from a form, you either use JavaScript (`fetch`/`axios`) or a hidden field method override (like `_method=DELETE`).

Flask's `methods` parameter accepts a list of method strings. The `HEAD` and `OPTIONS` methods are handled automatically by Flask — you don't need to declare them.

---

## ⚡ Syntax & First Usage

### Basic Routes

Let's write our first routes and test them immediately using Flask's built-in **test client** — no browser or running server needed!

The test client (`app.test_client()`) mimics HTTP requests so you can verify your routes work right inside a notebook.

In [ ]:
from flask import Flask

app = Flask(__name__)

@app.route('/')
def index():
    return '<h1>Welcome to the Home Page!</h1>'

@app.route('/about')
def about():
    return '<h1>About Us</h1><p>We build cool things with Flask.</p>'

@app.route('/contact')
def contact():
    return '<h1>Contact</h1><p>Reach us at hello@example.com</p>'

with app.test_client() as client:
    for url in ['/', '/about', '/contact']:
        response = client.get(url)
        print(f"GET {url} -> {response.status_code}")
        print(f"  Body: {response.data.decode()[:60]}...")
        print()


### URL Variables with Type Converters

Now let's make our routes **dynamic**. Notice how Flask automatically:
- Extracts the variable from the URL
- Converts it to the specified Python type
- Passes it as a keyword argument to the view function
- Returns 404 if the value doesn't match the type

In [ ]:
from flask import Flask

app = Flask(__name__)

# <string:name> - default, accepts any text without slashes
@app.route('/greet/<name>')
def greet(name):
    return f'Hello, {name}!'

# <int:id> - only matches integers; /user/abc returns 404 automatically
@app.route('/user/<int:id>')
def user_profile(id):
    return f'User profile #{id} (type: {type(id).__name__})'

# <float:score> - only matches floats (e.g. /score/9.5)
@app.route('/score/<float:score>')
def show_score(score):
    return f'Your score: {score:.2f} (type: {type(score).__name__})'

# <path:subpath> - matches everything including forward slashes
@app.route('/files/<path:subpath>')
def serve_file(subpath):
    return f'Serving file: {subpath}'

with app.test_client() as client:
    tests = [
        ('/greet/Alice',        200),
        ('/greet/Bob%20Smith',  200),
        ('/user/42',            200),
        ('/user/abc',           404),   # int converter rejects "abc"
        ('/score/9.75',         200),
        ('/files/docs/api/v2',  200),   # path converter accepts slashes
    ]
    for url, expected in tests:
        r = client.get(url)
        status = 'OK' if r.status_code == expected else 'FAIL'
        print(f"[{status}] GET {url} -> {r.status_code}: {r.data.decode()[:50]}")


---

## 🔬 Deep Dive

### `url_for()` — Build URLs, Don't Hardcode Them

Hardcoding URL strings like `'/user/42'` throughout your templates and code is a **maintenance trap**. The moment you rename a route, every hardcoded string breaks — silently.

`url_for()` solves this by generating URLs **from function names**. It's the single source of truth for your URL structure.

```python
url_for('view_function_name', param=value)
```

**Benefits:**
- **Rename a route** → only update the decorator, `url_for` calls still work
- **Handles encoding** → special characters are percent-encoded automatically (e.g. `Alice Brown` becomes `Alice%20Brown`)
- **`_external=True`** → returns an absolute URL including scheme and host (essential for emails, RSS feeds, OAuth callbacks)
- **Extra kwargs become query strings** → `url_for('search', q='flask', page=2)` produces `/search?q=flask&page=2`
- **Works in templates too** → `{{ url_for('static', filename='css/main.css') }}`

**When `url_for` raises an error:**
If you call `url_for('nonexistent_function')`, Flask raises a `BuildError` *immediately*. Compare that to a hardcoded string `/wrong-path` which silently returns 404 at runtime only when a user clicks that link. `url_for` turns a potential runtime mystery into an obvious startup-time error.

> ⚠️ **Must be called inside an application context.** In routes and templates this is automatic. In tests, wrap with `with app.app_context():`. In standalone scripts, push the context manually.

In [ ]:
from flask import Flask, url_for

app = Flask(__name__)

@app.route('/')
def index():
    return 'Home'

@app.route('/user/<int:id>')
def user_profile(id):
    return f'User {id}'

@app.route('/post/<int:post_id>/comment/<int:comment_id>')
def view_comment(post_id, comment_id):
    return f'Post {post_id}, Comment {comment_id}'

@app.route('/search')
def search():
    return 'Search results'

# url_for must be called inside an application context
with app.app_context():
    print("Basic URLs:")
    print(f"  index:        {url_for('index')}")
    print(f"  user #5:      {url_for('user_profile', id=5)}")

    print("\nMultiple variables:")
    print(f"  comment:      {url_for('view_comment', post_id=10, comment_id=3)}")

    print("\nQuery string params (extra kwargs become ?key=value):")
    print(f"  search:       {url_for('search', q='flask routing', page=2)}")

    print("\nAbsolute URLs (_external=True):")
    print(f"  user #7:      {url_for('user_profile', id=7, _external=True)}")
    print(f"  index:        {url_for('index', _external=True)}")


### HTTP Methods in Routes

By default Flask routes only respond to `GET`. Pass a `methods` list to accept others. A single view function can handle multiple methods — use `request.method` to branch behaviour.

In [ ]:
from flask import Flask, request, jsonify

app = Flask(__name__)

# GET only (default)
@app.route('/posts')
def list_posts():
    return jsonify({'posts': ['Post 1', 'Post 2', 'Post 3']})

# POST only - for creating resources
@app.route('/posts/new', methods=['POST'])
def create_post():
    data = request.get_json(silent=True) or {}
    title = data.get('title', 'Untitled')
    return jsonify({'created': True, 'title': title}), 201

# Handle GET and POST in the same view function
@app.route('/login', methods=['GET', 'POST'])
def login():
    if request.method == 'POST':
        data = request.get_json(silent=True) or {}
        username = data.get('username', '')
        return jsonify({'logged_in': True, 'user': username})
    return jsonify({'form': 'Please send POST with username/password'})

with app.test_client() as client:
    print("=== GET-only route ===")
    r = client.get('/posts')
    print(f"  GET  /posts -> {r.status_code}: {r.json}")

    print("\n=== POST-only route ===")
    r = client.post('/posts/new', json={'title': 'My First Post'},
                    content_type='application/json')
    print(f"  POST /posts/new -> {r.status_code}: {r.json}")

    print("\n=== Multi-method route ===")
    print(f"  GET  /login -> {client.get('/login').status_code}: {client.get('/login').json}")
    r = client.post('/login', json={'username': 'alice'},
                    content_type='application/json')
    print(f"  POST /login -> {r.status_code}: {r.json}")


### Redirects & the POST-Redirect-GET Pattern

A **redirect** tells the browser: "The resource you want is over there — go fetch it." Flask's `redirect()` function returns a `3xx` response with a `Location` header.

**301 vs 302 vs 303 — which code do you need?**

| Code | Name | Meaning | Browser caches? | Use case |
|------|------|---------|-----------------|----------|
| `301` | Moved Permanently | Resource has a new permanent home | Yes (browsers remember forever) | Renaming a URL permanently |
| `302` | Found | Temporary redirect | No | General-purpose temporary |
| `303` | See Other | After POST, go fetch this GET URL | No | **POST-Redirect-GET pattern** |

**Why 303 (not 302) for POST-Redirect-GET?**
303 explicitly means "I processed your POST; now do a GET to retrieve the result." Most browsers follow 302 redirects with GET too, but 303 makes the intent unambiguous. Always use 303 after a form submission.

**POST-Redirect-GET (PRG)** is a crucial web pattern:
1. User submits a `POST` form
2. Server processes data and **redirects** (303) to a `GET` page
3. Browser shows the result page

Without PRG, refreshing after a form submit re-sends the POST — potentially duplicating a payment, a comment, or a database record! The browser even warns with a dialog: *"This page is asking you to confirm that you want to send a form again."*

In [ ]:
from flask import Flask, redirect, url_for, request, jsonify

app = Flask(__name__)

@app.route('/')
def index():
    return jsonify({'page': 'home', 'message': 'Welcome!'})

@app.route('/old-home')
def old_home():
    return redirect(url_for('index'), code=301)

@app.route('/dashboard')
def dashboard():
    return jsonify({'page': 'dashboard'})

@app.route('/submit', methods=['POST'])
def submit():
    data = request.get_json(silent=True) or {}
    print(f"  [Server] Processing: {data}")
    # Redirect to a GET route so refresh won't resubmit
    return redirect(url_for('dashboard'), code=303)  # 303 See Other

with app.test_client() as client:
    print("=== Permanent redirect ===")
    r = client.get('/old-home')
    print(f"  GET /old-home -> {r.status_code} Location: {r.headers.get('Location')}")

    print("\n=== POST-Redirect-GET ===")
    r = client.post('/submit', json={'item': 'new blog post'},
                    content_type='application/json', follow_redirects=False)
    print(f"  POST /submit        -> {r.status_code} Location: {r.headers.get('Location')}")

    r = client.post('/submit', json={'item': 'new blog post'},
                    content_type='application/json', follow_redirects=True)
    print(f"  POST /submit (follow redirect) -> {r.status_code}: {r.json}")


 ### `abort()` — Stop Processing Immediately

`abort()` immediately stops request handling and returns an error response. Use it when a condition makes further processing pointless — unauthorized access, missing resource, server fault.

**Common status codes used with `abort()`:**

| Code | Name | When to use |
|------|------|-------------|
| `400` | Bad Request | The client sent invalid data (missing field, wrong format). Fix is on the client side. |
| `401` | Unauthorized | Authentication is required but was not provided (not logged in). |
| `403` | Forbidden | Authenticated but lacks permission (logged in but not an admin). |
| `404` | Not Found | The requested resource doesn't exist. |
| `405` | Method Not Allowed | Wrong HTTP method for this endpoint (Flask auto-raises this for you). |
| `409` | Conflict | Request conflicts with current state (e.g. creating a duplicate username). |
| `422` | Unprocessable Entity | Syntactically valid but semantically invalid (used in APIs). |
| `500` | Internal Server Error | Something went wrong server-side. |

**The difference between 401 and 403:**
- **401** means "Who are you? Please log in." The browser may show a login dialog.
- **403** means "I know who you are, but you're not allowed here." No login prompt.

You can pair `abort()` with `@app.errorhandler()` to return custom JSON or HTML instead of the default error page.

In [ ]:
from flask import Flask, abort, jsonify

app = Flask(__name__)

@app.errorhandler(404)
def not_found(e):
    return jsonify({'error': 'Not Found', 'message': str(e)}), 404

@app.errorhandler(403)
def forbidden(e):
    return jsonify({'error': 'Forbidden', 'message': "You don't have permission"}), 403

@app.errorhandler(500)
def server_error(e):
    return jsonify({'error': 'Internal Server Error', 'message': str(e)}), 500

USERS = {1: {'name': 'Alice', 'role': 'admin'}, 2: {'name': 'Bob', 'role': 'user'}}

@app.route('/user/<int:user_id>')
def get_user(user_id):
    user = USERS.get(user_id)
    if user is None:
        abort(404, description=f"No user with id={user_id}")
    return jsonify(user)

@app.route('/admin')
def admin_panel():
    is_admin = False  # Simulated auth check
    if not is_admin:
        abort(403)
    return jsonify({'panel': 'admin'})

with app.test_client() as client:
    print("=== 404 - user not found ===")
    r = client.get('/user/99')
    print(f"  GET /user/99 -> {r.status_code}: {r.json}")

    print("\n=== 200 - user exists ===")
    r = client.get('/user/1')
    print(f"  GET /user/1  -> {r.status_code}: {r.json}")

    print("\n=== 403 - forbidden ===")
    r = client.get('/admin')
    print(f"  GET /admin   -> {r.status_code}: {r.json}")


### RESTful URL Design

**REST** (Representational State Transfer) is a set of conventions for URL structure that makes APIs intuitive and predictable. The idea: URLs represent *resources* (nouns), HTTP methods represent *actions* (verbs).

| URL Pattern | Method | Action |
|-------------|--------|--------|
| `/users` | `GET` | List all users |
| `/users` | `POST` | Create a new user |
| `/users/<int:id>` | `GET` | Get a specific user |
| `/users/<int:id>` | `PUT` | Replace a user |
| `/users/<int:id>` | `PATCH` | Update part of a user |
| `/users/<int:id>` | `DELETE` | Delete a user |
| `/users/<int:id>/posts` | `GET` | List posts by this user |

> 💡 **Tip:** The URL (`/users/42`) identifies *what* you're acting on, the HTTP method says *how* — you never need `/deleteUser/42` or `/getUserPosts/42`.

In [ ]:
from flask import Flask, jsonify, request, abort

app = Flask(__name__)

users_db = {
    1: {'id': 1, 'name': 'Alice', 'email': 'alice@example.com'},
    2: {'id': 2, 'name': 'Bob',   'email': 'bob@example.com'},
}
posts_db = {
    1: [{'id': 1, 'title': 'Alice Post 1'}, {'id': 2, 'title': 'Alice Post 2'}],
    2: [{'id': 3, 'title': 'Bob Post 1'}],
}

@app.route('/users', methods=['GET'])
def list_users():
    return jsonify({'users': list(users_db.values())})

@app.route('/users', methods=['POST'])
def create_user():
    data = request.get_json(silent=True) or {}
    new_id = max(users_db) + 1
    users_db[new_id] = {'id': new_id, **data}
    return jsonify(users_db[new_id]), 201

@app.route('/users/<int:id>', methods=['GET'])
def get_user(id):
    user = users_db.get(id)
    if not user:
        abort(404)
    return jsonify(user)

@app.route('/users/<int:id>', methods=['DELETE'])
def delete_user(id):
    if id not in users_db:
        abort(404)
    del users_db[id]
    return '', 204  # No Content

@app.route('/users/<int:id>/posts', methods=['GET'])
def user_posts(id):
    if id not in users_db:
        abort(404)
    return jsonify({'user_id': id, 'posts': posts_db.get(id, [])})

with app.test_client() as client:
    print("GET  /users         ->", client.get('/users').json)
    print("GET  /users/1       ->", client.get('/users/1').json)
    print("GET  /users/1/posts ->", client.get('/users/1/posts').json)
    r = client.post('/users', json={'name': 'Carol', 'email': 'carol@example.com'},
                    content_type='application/json')
    print(f"POST /users         -> {r.status_code}: {r.json}")
    r = client.delete('/users/2')
    print(f"DEL  /users/2       -> {r.status_code} (204 = success, no body)")


---

### ⚖️ Comparison 1: Hardcoded URLs vs `url_for()`

It might seem easier to just write `'/user/42'` directly. Let's see exactly what breaks when you do that.

In [ ]:
from flask import Flask, url_for, redirect

app = Flask(__name__)

@app.route('/user/<int:id>')
def user_profile(id):
    return f'User {id}'

# APPROACH A: Hardcoded strings
@app.route('/hardcoded-redirect/<int:id>')
def hardcoded_redirect(id):
    # BAD: if we rename user_profile, this silently breaks
    return redirect(f'/user/{id}')

# APPROACH B: url_for()
@app.route('/safe-redirect/<int:id>')
def safe_redirect(id):
    # GOOD: rename user_profile and this still works
    return redirect(url_for('user_profile', id=id))

with app.app_context():
    print("=== Generated URLs ===")
    print(f"  url_for('user_profile', id=42)               -> {url_for('user_profile', id=42)}")
    print(f"  url_for('user_profile', id=42, _external=True) -> {url_for('user_profile', id=42, _external=True)}")

    print()
    print("=== What happens when a route is renamed? ===")
    print("  Hardcoded '/user/42'  -> nothing warns you; it silently 404s")
    print("  url_for('old_name')   -> BuildError raised at startup time (loud & clear)")
    print("  Result: url_for() turns a silent runtime bug into a startup error [OK]")

with app.test_client() as client:
    r = client.get('/safe-redirect/7', follow_redirects=False)
    print(f"\nGET /safe-redirect/7 -> {r.status_code} Location: {r.headers['Location']}")


### ⚖️ Comparison 2: GET vs POST

Both `GET` and `POST` send data to the server — but in very different ways with very different implications.

In [ ]:
from flask import Flask, request, jsonify

app = Flask(__name__)

# GET: data travels in the URL query string
@app.route('/search')
def search():
    query = request.args.get('q', '')
    page  = request.args.get('page', 1, type=int)
    return jsonify({
        'method': 'GET',
        'query':  query,
        'page':   page,
        'note':   'URL is bookmarkable & shareable'
    })

# POST: data travels in the request body
@app.route('/login', methods=['POST'])
def login():
    data     = request.get_json(silent=True) or {}
    username = data.get('username', '')
    return jsonify({
        'method':   'POST',
        'username': username,
        'note':     'Password NOT visible in URL or browser history'
    })

with app.test_client() as client:
    print("=== GET /search ===")
    r = client.get('/search?q=flask+routing&page=2')
    print(f"  {r.json}")
    print("  ^ Query params visible in URL - great for search, bad for passwords\n")

    print("=== POST /login ===")
    r = client.post('/login', json={'username': 'alice', 'password': 's3cr3t!'},
                    content_type='application/json')
    print(f"  {r.json}")
    print("  ^ Credentials stay in request body - never appear in browser history")

print('''
+-------------+--------------------------+--------------------------+
|             | GET                      | POST                     |
+-------------+--------------------------+--------------------------+
| Data in     | URL query string         | Request body             |
| Bookmarkable| Yes                      | No                       |
| Safe repeat | Yes (no side effects)    | No (creates/changes)     |
| Use for     | Search, filters, pages   | Login, create, update    |
+-------------+--------------------------+--------------------------+''')


---

## 🧪 What If? Experimentation

This is where things get interesting. Let's deliberately poke Flask in unexpected ways and observe exactly what happens.

### ❓ Q1: What if you visit a URL that matches no route?

Flask's router scans its URL map. If no pattern matches, what does it do?

In [ ]:
from flask import Flask, jsonify

app = Flask(__name__)

@app.errorhandler(404)
def not_found(e):
    return jsonify({'error': '404 Not Found', 'message': str(e)}), 404

@app.route('/')
def index():
    return 'Home'

@app.route('/about')
def about():
    return 'About'

with app.test_client() as client:
    print("=== Known routes ===")
    print(f"  GET /       -> {client.get('/').status_code}")
    print(f"  GET /about  -> {client.get('/about').status_code}")

    print("\n=== Unknown routes ===")
    for bad_url in ['/missing', '/About', '/about/', '/home/page/one']:
        r = client.get(bad_url)
        print(f"  GET {bad_url:<25} -> {r.status_code}: {r.json}")

print('''
Key observations:
  * Flask returns 404 for any URL not in the URL map
  * /About != /about -- routes are case-sensitive by default
  * Trailing slash behaviour depends on how the route is defined
  * Your @errorhandler(404) controls the response format''')


### ❓ Q2: What if you send POST to a GET-only route?

Flask enforces the `methods` list strictly. What happens when the method isn't allowed?

In [ ]:
from flask import Flask, jsonify

app = Flask(__name__)

@app.errorhandler(405)
def method_not_allowed(e):
    return jsonify({
        'error': '405 Method Not Allowed',
        'allowed': sorted(e.valid_methods) if e.valid_methods else []
    }), 405

@app.route('/articles')                           # GET only (default)
def list_articles():
    return jsonify({'articles': ['Article 1', 'Article 2']})

@app.route('/articles/create', methods=['POST'])  # POST only
def create_article():
    return jsonify({'created': True}), 201

with app.test_client() as client:
    print("=== Correct methods ===")
    print(f"  GET  /articles        -> {client.get('/articles').status_code}")
    print(f"  POST /articles/create -> {client.post('/articles/create').status_code}")

    print("\n=== Wrong methods ===")
    r = client.post('/articles')
    print(f"  POST /articles        -> {r.status_code}: {r.json}")
    r = client.get('/articles/create')
    print(f"  GET  /articles/create -> {r.status_code}: {r.json}")
    r = client.delete('/articles')
    print(f"  DEL  /articles        -> {r.status_code}: {r.json}")

print('''
Key takeaway:
  Flask returns 405 Method Not Allowed (not 404) -- the URL exists,
  but the method isn't permitted. The Allow header lists valid methods.''')


### ❓ Q3: What if two URL patterns could both match the same request?

Consider `/users/<int:id>` and `/users/profile`. The URL `/users/profile` could match both — but a type converter narrows it. Let's verify Flask's rule.

In [ ]:
from flask import Flask, jsonify

app = Flask(__name__)

# Flask matches routes by specificity, not just registration order.
# A static segment ('profile') wins over a variable segment for the same position.

@app.route('/users/<int:id>')       # matches /users/42, /users/1, etc.
def get_user(id):
    return jsonify({'route': 'get_user', 'id': id})

@app.route('/users/profile')        # static -- matches ONLY /users/profile
def user_profile():
    return jsonify({'route': 'user_profile', 'page': 'profile'})

@app.route('/users/<string:name>')  # matches any string (fallback)
def user_by_name(name):
    return jsonify({'route': 'user_by_name', 'name': name})

with app.test_client() as client:
    tests = [
        '/users/42',       # int -> get_user wins
        '/users/profile',  # static -> user_profile wins (NOT user_by_name)
        '/users/alice',    # string -> user_by_name
        '/users/123abc',   # not int, not 'profile' -> user_by_name
    ]
    for url in tests:
        r = client.get(url)
        print(f"  GET {url:<20} -> {r.status_code}: {r.json}")

print('''
Flask's URL routing rules:
  1. Static segments beat variable segments for the same position
  2. Type converters narrow matches ('<int:id>' won't match 'profile')
  3. More specific rules win; equally specific routes raise an error at startup''')


---

## 🚀 Real-World Project Link

Everything in this chapter feeds directly into a real Flask application. In our **Flask Blog** project, we'll use exactly these patterns:

| Route | Method | View Function | Purpose |
|-------|--------|---------------|---------|
| `/` | `GET` | `index` | Homepage — list recent posts |
| `/post/<int:id>` | `GET` | `view_post` | Display a single blog post |
| `/post/new` | `GET`, `POST` | `new_post` | Show form / submit new post |
| `/post/<int:id>/edit` | `GET`, `PUT` | `edit_post` | Edit existing post |
| `/post/<int:id>/delete` | `POST` | `delete_post` | Delete (POST + redirect) |
| `/user/<username>` | `GET` | `user_profile` | Public user profile |
| `/admin/` | `GET` | `admin_panel` | Admin dashboard |

Every route follows the RESTful principles from this chapter:
- URLs name **resources** (`/post/42`, not `/getPost?id=42`)
- HTTP methods describe **actions** (`DELETE /post/42`, not `/deletePost/42`)
- `url_for()` is used everywhere — no hardcoded strings
- POST forms always redirect after success (POST-Redirect-GET)

> 💡 **Tip:** Sketch your entire URL map on paper before writing any view functions. A well-designed URL map makes the rest of your app fall into place.

---

## 📋 Chapter Summary & Cheat Sheet

Here's everything from this chapter in one place — save this as your routing reference.

### Key Concepts

| Concept | What It Does |
|---------|-------------|
| `@app.route('/path')` | Registers a URL → function mapping |
| `<variable>` | Dynamic URL segment (string by default) |
| `<int:id>`, `<float:x>`, `<path:p>`, `<uuid:t>` | Type converters — restrict + auto-convert |
| `url_for('func_name', param=val)` | Generate URL from function name (never hardcode!) |
| `redirect(url)` | Return 3xx response pointing browser elsewhere |
| `abort(code)` | Stop handling and return error response |
| `@app.errorhandler(code)` | Custom error response for a given HTTP code |
| `methods=['GET','POST']` | Which HTTP methods this route accepts |
### HTTP Status Codes You'll Use Most

| Code | Name | When |
|------|------|------|
| `200` | OK | Success |
| `201` | Created | Resource created (POST) |
| `204` | No Content | Success, no body (DELETE) |
| `301` | Moved Permanently | Old URL → new URL forever |
| `302` / `303` | Found / See Other | Temporary redirect (PRG) |
| `400` | Bad Request | Client sent bad data |
| `403` | Forbidden | No permission |
| `404` | Not Found | URL doesn't exist |
| `405` | Method Not Allowed | Wrong HTTP method |
| `500` | Internal Server Error | Something crashed server-side |

In [ ]:
# Flask Routing Cheat Sheet -- copy & paste ready!

from flask import Flask, url_for, redirect, abort, request, jsonify

app = Flask(__name__)

# 1. Basic route
@app.route('/')
def index():
    return 'Hello, World!'

# 2. Type converters
@app.route('/item/<int:id>')
def get_item(id): return f'item {id}'

@app.route('/rate/<float:score>')
def rate(score): return f'score {score}'

@app.route('/file/<path:subpath>')
def serve(subpath): return f'file: {subpath}'

# 3. HTTP methods
@app.route('/resource', methods=['GET', 'POST'])
def resource():
    if request.method == 'POST':
        return jsonify(request.get_json()), 201
    return jsonify({'items': []})

# 4. url_for()
with app.app_context():
    print(url_for('index'))                           # /
    print(url_for('get_item', id=5))                  # /item/5
    print(url_for('get_item', id=5, _external=True))  # http://localhost/item/5
    print(url_for('resource', filter='all', page=2))  # /resource?filter=all&page=2

# 5. redirect()
@app.route('/old')
def old(): return redirect(url_for('index'), 301)

@app.route('/form', methods=['POST'])
def handle_form():
    return redirect(url_for('index'), 303)  # POST-Redirect-GET

# 6. abort() + error handler
@app.errorhandler(404)
def not_found(e): return jsonify({'error': str(e)}), 404

@app.route('/must-exist/<int:id>')
def must_exist(id):
    if id > 100: abort(404)
    return jsonify({'id': id})

print("Cheat sheet loaded successfully!")


---

## 💪 Practice Prompts

These challenges reinforce everything in this chapter. Try each one before looking for hints!

---

### 🏋️ Challenge 1: Design a Blog URL Scheme

**Task:** Without writing any Python, sketch the complete URL map for a blog application. For each route, define:
- The URL pattern (with type converters)
- The HTTP methods it accepts
- The view function name
- What it does

Your blog needs: list posts, view a post, create a post, edit a post, delete a post, view author profiles, tag pages, and an RSS feed.

<details>
<summary>💡 Hint (expand when ready)</summary>

Think in terms of resources: `posts`, `users`, `tags`. A resource typically needs 5 routes (list, detail, create, update, delete). Don't forget edge cases: RSS feed might use `/feed.xml` or `/rss`.

</details>

---

### 🏋️ Challenge 2: Calculator API with URL Variables

**Task:** Build a calculator API using URL variables — no JSON body needed!

Required routes:
- `GET /calc/add/<int:a>/<int:b>` → `{"result": 7}` for `/calc/add/3/4`
- `GET /calc/sub/<int:a>/<int:b>` → subtraction
- `GET /calc/mul/<int:a>/<int:b>` → multiplication
- `GET /calc/div/<float:a>/<float:b>` → division (handle division by zero!)

Test it with `app.test_client()`.

<details>
<summary>💡 Hint (expand when ready)</summary>

Use `<int:a>` for add/sub/mul. Use `<float:a>` for div so it handles `3/2 = 1.5`. For division by zero, `abort(400)` with a helpful message.

</details>

---

### 🏋️ Challenge 3: POST-Redirect-GET Pattern

**Task:** Implement a "guestbook" with POST-Redirect-GET:

1. `GET /guestbook` → Return list of all messages (as JSON)
2. `POST /guestbook` → Accept `{"name": "...", "message": "..."}`, store it, then **redirect** to `GET /guestbook`
3. Verify that after posting, the client ends up on `GET /guestbook` with the new message included
4. Verify that repeating the same GET request doesn't duplicate the message

<details>
<summary>💡 Hint (expand when ready)</summary>

Store messages in a module-level list. Use `redirect(url_for('list_guestbook'), 303)` after the POST. Test with `follow_redirects=True` on the POST call, then make a second GET to confirm no duplication.

</details>

In [ ]:
# Challenge 2 Starter: Calculator API
# Fill in the TODOs and run!

from flask import Flask, jsonify, abort

app = Flask(__name__)

# TODO: implement /calc/add/<int:a>/<int:b>
# TODO: implement /calc/sub/<int:a>/<int:b>
# TODO: implement /calc/mul/<int:a>/<int:b>
# TODO: implement /calc/div/<float:a>/<float:b>  -- handle division by zero!

# with app.test_client() as client:
#     print(client.get('/calc/add/3/4').json)    # {'result': 7}
#     print(client.get('/calc/div/10/3').json)   # {'result': 3.333...}
#     print(client.get('/calc/div/5/0').json)    # 400 error

print("Implement the routes above, then uncomment the test block!")


In [ ]:
# Challenge 3 Starter: POST-Redirect-GET Guestbook

from flask import Flask, request, redirect, url_for, jsonify

app = Flask(__name__)
messages = []  # In-memory store

# TODO: implement GET /guestbook  -> return jsonify({'messages': messages})
# TODO: implement POST /guestbook -> append to messages, then redirect (303)

# with app.test_client() as client:
#     print("Before:", client.get('/guestbook').json)
#
#     r = client.post('/guestbook',
#                     json={'name': 'Alice', 'message': 'Hello!'},
#                     content_type='application/json',
#                     follow_redirects=True)
#     print("After POST:", r.json)
#
#     # GET again -- message should appear exactly once
#     print("Second GET:", client.get('/guestbook').json)

print("Implement the routes above, then uncomment the test block!")


---

## 🎉 You've Mastered Routing!

Here's what you can now do:

- ✅ Define static and dynamic routes with `@app.route()`
- ✅ Use type converters to validate and convert URL variables
- ✅ Generate safe, maintainable URLs with `url_for()`
- ✅ Handle different HTTP methods in the same or separate view functions
- ✅ Use `redirect()` and the POST-Redirect-GET pattern
- ✅ Return proper error codes with `abort()` and custom error handlers
- ✅ Design RESTful URL structures

**Up next → Chapter 3: Templates with Jinja2**
Your routes return plain strings right now. In Chapter 3 we'll return beautiful HTML by connecting routes to **Jinja2 templates** — Flask's powerful templating engine.

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='1. getting_started.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Chapter 1: Getting Started</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='3. templates_with_jinja2.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 3: Jinja2 Templates →</a>
</div>

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='1. getting_started.ipynb' style='font-weight:bold; font-size:1.05em;'>&larr; Previous</a>
  <a href='../TOC.md' style='font-weight:bold; font-size:1.05em; text-align:center;'>Table of Contents</a>
  <a href='3. templates_with_jinja2.ipynb' style='font-weight:bold; font-size:1.05em;'>Next &rarr;</a>
</div>